# Suspicious outlet-photo detection — GPU run

Runs the `outlet-audit` pipeline on a GPU (Tesla T4 or similar).

**Steps**

1. **Check the GPU** — `nvidia-smi` must list a device. No device = runtime has no GPU; switch runtime to GPU first.
2. **Get the code + dataset** — zip the repo locally (`zip -r IM-Assesment.zip IM-Assesment -x 'IM-Assesment/.venv/*' 'IM-Assesment/cache/*' 'IM-Assesment/.git/*' 'IM-Assesment/results/*'`), upload it to the root of Google Drive. The cell mounts Drive, unzips to `/content/IM-Assesment` and `%cd`s into it (`%cd` persists across cells, `!cd` does not).
3. **Install the package** — `pip install -e .` installs `outlet-audit` from this repo. The torch check must print `True`; if it prints `False`, the install pulled a CPU-only torch. Fix with:
   `pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128`
4. **Run the pipeline** — `--device cuda` moves DINOv2, CLIP and EasyOCR to the GPU. `--batch-size 64` is safe on a 16 GB T4. First run downloads model weights (~1 GB). The SIFT/RANSAC geometry stage is CPU-only and unaffected by the GPU.

Outputs land in `results/`: per-outlet JSON/CSV plus an HTML report (`--report`). Expects `dataset/<outlet_id>/*.jpg`.


In [2]:
!nvidia-smi

Fri Sep  4 10:23:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/IM-Assesment.zip -d /content/repo
import glob, os
%cd {os.path.dirname(glob.glob('/content/repo/**/pyproject.toml', recursive=True)[0])}
!ls


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/repo
AI_Engineer_Assignment_Suspicious_Photo_Detection.docx.pdf
config.yaml
dataset
pyproject.toml
README.md
scripts
src
suspicious_photo_detection.ipynb
tests
uv.lock
WRITEUP.md


In [8]:
!pip install -q -e .
import torch; print(torch.__version__, torch.cuda.is_available())


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 76.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 MB 13.1 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 110.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 30.4 MB/s eta 0:00:00
  Building editable for outlet-audit (pyproject.

In [10]:
!outlet-audit run --data dataset --out results --config config.yaml --device cuda --batch-size 64 --report


2026-09-04 11:34:27,288 INFO outlet_audit: device=cuda
2026-09-04 11:34:55,168 INFO outlet_audit: 159 outlets, 2042 files, 0 unreadable
2026-09-04 11:34:58.565307: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-04 11:35:05,891 INFO outlet_audit.embed: facebook__dinov2-base_294x224_cls: 2042 cached, 0 to embed
2026-09-04 11:35:06,439 INFO outlet_audit.pipeline: consensus LR: floor=0.271 legit med=0.731 foreign med=0.393
^C
